<a href="https://colab.research.google.com/github/empeleleb-code/Elijah_gitHub/blob/main/ICD_5021_ASSIGNMENT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Project Title and Overview:TymeBank Real-Time Streaming ETL Pipeline
###Author: ELIJAH MPELELE B
Module: ICD 5021

##Architecture Overview

This project implements a real-time ETL (Extract, Transform, Load) pipeline using Apache Kafka and PySpark Structured Streaming. It follows the Medallion Architecture (Bronze → Silver → Gold) and uses Delta Lake for reliable, ACID-compliant storage.

##Data Flow:

1. Source: A Python script simulates real-time transaction events by streaming a CSV dataset into a Kafka topic.

2. Bronze Layer: Raw JSON messages are ingested from Kafka and stored as Delta tables (immutable audit trail).

3. Silver Layer: Data is parsed, cleaned, deduplicated, and enriched. A Watermark is applied to handle late-arriving data.

4. Gold Layer: Business-level aggregations (e.g., transaction volumes by city and merchant) are calculated using windowed operations.

5. Sink: In production, these Delta tables sink to Azure Data Lake Gen 2 (ADLS). For this Colab demonstration, local Colab storage is used to simulate the ADLS file system.



 # Phase 1 - Infrastructure Setup
 ## Kafka Infrastructure Setup (KRaft Mode)
 ### Setup Java & Kafka
In this phase, we install Java and Apache Kafka. We start Kafka using KRaft (Kafka Raft), which eliminates the need for ZooKeeper, simplifying the architecture. Finally, we create the Kafka topic where our transaction data will stream.



In [1]:
# 1. Update package index and install Java
print("Updating package lists...")
!apt-get update -qq
print("Installing Java...")
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

# 2. Download and extract Kafka
print("Downloading Kafka...")
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.12-3.6.1.tgz
!tar -xzf kafka_2.12-3.6.1.tgz
%cd kafka_2.12-3.6.1

# 3. Generate Cluster ID (Robustly)
# The JVM may print warnings to stderr; we capture them to avoid contaminating the ID.
import subprocess
print("Generating Cluster ID...")
result = subprocess.run(
    ["bin/kafka-storage.sh", "random-uuid"],
    capture_output=True, text=True
)
# The UUID is on the last line of stdout
cluster_id = result.stdout.strip().splitlines()[-1]
print(f"Generated Cluster ID: {cluster_id}")

# 4. Format storage and start Kafka
print("Formatting storage...")
!bin/kafka-storage.sh format -t {cluster_id} -c config/kraft/server.properties

print("Starting Kafka in KRaft mode...")
!nohup bin/kafka-server-start.sh config/kraft/server.properties > /content/kafka.log 2>&1 &

# 5. Wait for Kafka to boot and verify it's running
import time
print("Waiting 20 seconds for Kafka to initialize...")
time.sleep(20)

# Verify the broker is running by listing topics
print("Verifying Kafka is running...")
!bin/kafka-topics.sh --list --bootstrap-server localhost:9092

# 6. Create the topic
print("Creating topic...")
!bin/kafka-topics.sh --create --topic tyme-bank-transactions --bootstrap-server localhost:9092 --partitions 1 --replication-factor 1

print("✅ Kafka setup complete and topic created!")

Updating package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Installing Java...
/content/kafka_2.12-3.6.1
Generating Cluster ID...
Generated Cluster ID: NzRuLgb_TsiiW1HXQyCoTA
Formatting storage...
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Log directory /tmp/kraft-combined-logs is already formatted. Use --ignore-formatted to ignore this directory and format the others.
Starting Kafka in KRaft mode...
Waiting 20 seconds for Kafka to initialize...
Verifying Kafka is running...
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgrou

 # Phase 2 - Python Dependencies
## Install Python Libraries
We need PySpark for distributed streaming, Delta Lake for storage, and a Kafka client for Python.
Note: We are using kafka-python-ng instead of kafka-python to ensure compatibility with Python 3.12+ (which is currently used in Google Colab).

In [2]:
print("Installing Python dependencies...")
# Uninstall incompatible version and install the maintained fork
!pip uninstall -y kafka-python > /dev/null 2>&1
!pip install -q kafka-python-ng

# Install PySpark and Delta Lake
!pip install -q pyspark==3.5.0 delta-spark==3.1.0

print("✅ Libraries installed successfully!")

Installing Python dependencies...
✅ Libraries installed successfully!


 # Phase 3 - Data Preparation
## Data Ingestion Preparation
We will simulate the dataset. Since we  have the full Synthetic_TymeBank_Host_Data_500_Rows.csv uploaded to Colab,we  adjust the file path in Phase 4. For ease of execution, this cell generates a sample CSV with the exact structure of the TymeBank dataset.

# Phase 4 - Kafka Producer
## Simulating Real-Time Events (Kafka Producer)
This script acts as the source system. It reads the CSV row-by-row, converts each row into a JSON object, and streams it to the Kafka topic we created in Phase 1.

In [3]:
import json
import time
from kafka import KafkaProducer
import pandas as pd

print("Starting Kafka Producer...")
producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

# Read the dataset
df = pd.read_csv('/content/Synthetic_TymeBank_Host_Data_500_Rows.csv', encoding='utf-8-sig')

# Stream rows with a short delay to simulate real-time events
for index, row in df.iterrows():
    message = row.where(pd.notnull(row), None).to_dict()
    producer.send('tyme-bank-transactions', value=message)
    print(f"Sent row: {index + 1} | ACCT_NO: {message['ACCT_NO']} | Amount: {message['ACTAMT']}")
    time.sleep(0.1) # 100ms delay

producer.flush()
print("\n✅ All rows successfully produced to Kafka!")

Starting Kafka Producer...
Sent row: 1 | ACCT_NO: 99000000157 | Amount: 50000
Sent row: 2 | ACCT_NO: 99000000027 | Amount: 18597
Sent row: 3 | ACCT_NO: 99000000102 | Amount: 1299
Sent row: 4 | ACCT_NO: 99000000007 | Amount: 100000
Sent row: 5 | ACCT_NO: 99000000038 | Amount: 25000
Sent row: 6 | ACCT_NO: 99000000073 | Amount: 84689
Sent row: 7 | ACCT_NO: 99000000164 | Amount: 5999
Sent row: 8 | ACCT_NO: 99000000092 | Amount: 39999
Sent row: 9 | ACCT_NO: 99000000142 | Amount: 100000
Sent row: 10 | ACCT_NO: 99000000139 | Amount: 995
Sent row: 11 | ACCT_NO: 99000000120 | Amount: 10000
Sent row: 12 | ACCT_NO: 99000000137 | Amount: 74886
Sent row: 13 | ACCT_NO: 99000000125 | Amount: 38432
Sent row: 14 | ACCT_NO: 99000000050 | Amount: 4999
Sent row: 15 | ACCT_NO: 99000000110 | Amount: 17500
Sent row: 16 | ACCT_NO: 99000000155 | Amount: 100000
Sent row: 17 | ACCT_NO: 99000000085 | Amount: 2500
Sent row: 18 | ACCT_NO: 99000000098 | Amount: 29904
Sent row: 19 | ACCT_NO: 99000000023 | Amount: 250

# Phase 5 - Spark Session Configuration
## PySpark Session & ADLS Gen 2 Configuration
We initialize a single SparkSession that includes the Delta Lake extensions and the Kafka connector.
For production, the LAKEHOUSE_BASE path points to Azure Data Lake Gen 2. In Colab, we simulate this using local storage.



In [4]:
import pyspark
from delta import *

# ==============================================================================
# AZURE ADLS GEN 2 CONFIGURATION (UNCOMMENT & FILL IN FOR AZURE)
# ==============================================================================
# ADLS_ACCOUNT_NAME = "yourstorageaccount"
# ADLS_CONTAINER = "yourcontainer"
# ADLS_KEY = "your-storage-account-key"
# LAKEHOUSE_BASE = f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/tyme-bank"
# ==============================================================================

# FOR COLAB TESTING: Local storage simulating ADLS Gen 2
LAKEHOUSE_BASE = "/content/delta_lake/tyme-bank"

builder = pyspark.sql.SparkSession.builder.appName("TymeBankKafkaETL") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.shuffle.partitions", "1")

    # UNCOMMENT THIS LINE FOR AZURE:
    # .config(f"fs.azure.account.key.{ADLS_ACCOUNT_NAME}.dfs.core.windows.net", ADLS_KEY)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"]
).getOrCreate()

print("✅ Spark Session Initialized successfully!")

✅ Spark Session Initialized successfully!


# Phase 6 - Bronze Layer
## Bronze Layer (Raw Ingestion)
### Bronze Streaming Query
The Bronze layer ingests the raw JSON payload from Kafka without any transformations. We add kafka_timestamp and processing_timestamp for auditing and lineage tracking. The data is written to a Delta Table using append mode.



In [5]:
from pyspark.sql.functions import col, current_timestamp

# Read from Kafka
raw_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "tyme-bank-transactions") \
    .option("startingOffsets", "earliest") \
    .load()

# Add metadata
bronze_df = raw_df.select(
    col("value").cast("string").alias("raw_payload"),
    col("timestamp").alias("kafka_timestamp"),
    current_timestamp().alias("processing_timestamp")
)

# Write to Bronze Delta Table
bronze_query = bronze_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{LAKEHOUSE_BASE}/_checkpoints/bronze") \
    .start(f"{LAKEHOUSE_BASE}/bronze/transactions")

print("✅ Bronze Streaming Query Started!")

✅ Bronze Streaming Query Started!


# Phase 7 - Silver Layer
## Silver Layer (Cleansed, Enriched & Watermarked)
### Silver Streaming Query
This is the core transformation layer. We parse the JSON, cast the event_time to a proper TimestampType, and apply a 10-minute Watermark. The watermark handles late-arriving data and ensures stateful operations (like dropDuplicates) do not grow state indefinitely.

Note: We explicitly define the bronze_schema to prevent the AnalysisException that occurs when a streaming Delta table hasn't fully registered its schema yet.



In [6]:
from pyspark.sql.functions import from_json, to_timestamp, concat_ws, col, expr
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# 1. Define schema for the JSON payload
payload_schema = StructType([
    StructField("HPAN", StringType(), True), StructField("ACCT_NO", StringType(), True),
    StructField("ACCTBAL", DoubleType(), True), StructField("ACTAMT", DoubleType(), True),
    StructField("RESP", StringType(), True), StructField("RESPCODE", StringType(), True),
    StructField("TRANS_TYPE", StringType(), True), StructField("ADDRESS_CITY", StringType(), True),
    StructField("MERCHANT", StringType(), True), StructField("SWITCH_DATE", StringType(), True),
    StructField("SWITCH_TIME", StringType(), True)
])

# 2. Read from Bronze stream
bronze_stream = spark.readStream \
    .format("delta") \
    .load(f"{LAKEHOUSE_BASE}/bronze/transactions")

# 3. Parse, Clean, Deduplicate, Shift Time, and Watermark
silver_df = bronze_stream \
    .select(from_json(col("raw_payload"), payload_schema).alias("data")) \
    .select("data.*") \
    .withColumn("event_time",
                to_timestamp(concat_ws(" ", col("SWITCH_DATE"), col("SWITCH_TIME")), "yyyyMMdd HHmmss") - expr("INTERVAL 2 YEARS")) \
    .filter(col("ACCT_NO").isNotNull()) \
    .dropDuplicates(["HPAN", "ACCT_NO", "event_time"]) \
    .withWatermark("event_time", "10 minutes")

# 4. Write to Silver Delta Table
silver_query = silver_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{LAKEHOUSE_BASE}/_checkpoints/silver") \
    .start(f"{LAKEHOUSE_BASE}/silver/transactions")

print("✅ Silver Streaming Query Started with Watermarking!")

✅ Silver Streaming Query Started with Watermarking!


# The Fix: Verify Silver Before Running Gold
we must confirm that the Silver table actually has data before ## starting the Gold stream.

###Step 1: Stop any running streams
If we have any active Spark streams from previous attempts, we run the following code to clear them:

In [7]:
for s in spark.streams.active:
    s.stop()
print("Stopped all active streams.")

Stopped all active streams.


###Step 2: Run a Batch Read on Silver
We create a new cell and run the following code to check if the Silver table exists and has data:

In [8]:
try:
    print("Checking Silver table...")
    spark.read.format("delta").load(f"{LAKEHOUSE_BASE}/silver/transactions").show(5)
    print("✅ Silver table exists and has data!")
except Exception as e:
    print("❌ ERROR: Silver table is empty or doesn't exist.")
    print("You must run the Silver Streaming Query and wait for it to process data before running Gold.")

Checking Silver table...
+----+-------+-------+------+----+--------+----------+------------+--------+-----------+-----------+----------+
|HPAN|ACCT_NO|ACCTBAL|ACTAMT|RESP|RESPCODE|TRANS_TYPE|ADDRESS_CITY|MERCHANT|SWITCH_DATE|SWITCH_TIME|event_time|
+----+-------+-------+------+----+--------+----------+------------+--------+-----------+-----------+----------+
+----+-------+-------+------+----+--------+----------+------------+--------+-----------+-----------+----------+

✅ Silver table exists and has data!


###Step 3: If the check fails, fix the Silver layer
If the check failed, it means the Silver stream didn't run or didn't write data. We therefore, tried to use the corrected Silver code with the INTERVAL 2 YEARS fix.


In [9]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# 1. Define the exact schema of the Silver table
silver_schema = StructType([
    StructField("HPAN", StringType(), True), StructField("ACCT_NO", StringType(), True),
    StructField("ACCTBAL", DoubleType(), True), StructField("ACTAMT", DoubleType(), True),
    StructField("RESP", StringType(), True), StructField("RESPCODE", StringType(), True),
    StructField("TRANS_TYPE", StringType(), True), StructField("ADDRESS_CITY", StringType(), True),
    StructField("MERCHANT", StringType(), True), StructField("SWITCH_DATE", StringType(), True),
    StructField("SWITCH_TIME", StringType(), True), StructField("event_time", TimestampType(), True)
])

# 2. Create an empty DataFrame with this schema
empty_df = spark.createDataFrame([], silver_schema)

# 3. Write it as a Delta table (this creates the _delta_log)
empty_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{LAKEHOUSE_BASE}/silver/transactions")

print("✅ Empty Silver table created with schema. You can now run the Gold code!")

✅ Empty Silver table created with schema. You can now run the Gold code!


# Phase 8 - Gold Layer
## Gold Layer (Business Aggregations)
 ### Gold Streaming Query
The Gold layer provides business value. We read from the Silver table and perform a 5-minute windowed aggregation to calculate total transaction amounts and counts grouped by ADDRESS_CITY, MERCHANT, and RESP (response status).
We use outputMode("complete") because windowed aggregations require Spark to output the full updated result table.



In [10]:
from pyspark.sql.functions import window, sum, count, avg, col

# 1. Read from Silver stream
silver_stream = spark.readStream.format("delta").load(f"{LAKEHOUSE_BASE}/silver/transactions")

# 2. Perform windowed aggregations (5-minute window)
gold_df = silver_stream \
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("ADDRESS_CITY"),
        col("MERCHANT"),
        col("RESP")
    ) \
    .agg(
        count("*").alias("transaction_count"),
        sum("ACTAMT").alias("total_amount"),
        avg("ACTAMT").alias("avg_amount")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("ADDRESS_CITY"),
        col("MERCHANT"),
        col("RESP"),
        col("transaction_count"),
        col("total_amount"),
        col("avg_amount")
    )

# 3. Write to Gold Delta Table
gold_query = gold_df.writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", f"{LAKEHOUSE_BASE}/_checkpoints/gold") \
    .start(f"{LAKEHOUSE_BASE}/gold/merchant_summary")

print("✅ Gold Streaming Query Started successfully!")

✅ Gold Streaming Query Started successfully!


#Phase 9 - Execution & Monitoring
## Stop Streams

Streaming queries are designed to run indefinitely (24/7). However, in a Google Colab environment, letting them run forever will consume all available memory and eventually crash the notebook. This phase acts as our orchestrator: it lets the pipeline run for 30 seconds to ensure all data is processed, and then gracefully stops all active streaming queries.


In [11]:
import time

print("Letting the pipeline run for 30 seconds to process data...")
time.sleep(30)

print("Stopping active streams...")
# Stop the specific queries we started
try:
    bronze_query.stop()
    silver_query.stop()
    gold_query.stop()
    print("✅ All three streams (Bronze, Silver, Gold) stopped successfully.")
except Exception as e:
    print(f"Note: One or more streams may have already stopped. Details: {e}")

Letting the pipeline run for 30 seconds to process data...
Stopping active streams...
✅ All three streams (Bronze, Silver, Gold) stopped successfully.


#Phase 10: Data Validation & Results

Now that the streams are stopped, the Delta tables are finalized. We can read them in batch mode (using spark.read instead of spark.readStream) to verify that our ETL pipeline transformed the raw JSON into business-ready aggregations.
1. Bronze: Should show raw JSON strings.
2. Silver: Should show parsed columns, timestamped, and shifted back 2 years.
3. Gold: Should show the 5-minute windowed business aggregations.

In [12]:
print("======================================================")
print("🥉 BRONZE LAYER SAMPLE (Raw JSON from Kafka)")
print("======================================================")
spark.read.format("delta").load(f"{LAKEHOUSE_BASE}/bronze/transactions") \
    .select("raw_payload", "kafka_timestamp") \
    .show(3, truncate=60)

print("\n======================================================")
print("🥈 SILVER LAYER SAMPLE (Cleaned & Deduplicated)")
print("======================================================")
spark.read.format("delta").load(f"{LAKEHOUSE_BASE}/silver/transactions") \
    .select("ACCT_NO", "ADDRESS_CITY", "MERCHANT", "ACTAMT", "event_time") \
    .show(10, truncate=False)

print("\n======================================================")
print("🥇 GOLD LAYER (Business Aggregations)")
print("======================================================")
spark.read.format("delta").load(f"{LAKEHOUSE_BASE}/gold/merchant_summary") \
    .select("window_start", "window_end", "ADDRESS_CITY", "MERCHANT", "RESP", "transaction_count", "total_amount") \
    .orderBy("window_start") \
    .show(truncate=False)

🥉 BRONZE LAYER SAMPLE (Raw JSON from Kafka)
+------------------------------------------------------------+-----------------------+
|                                                 raw_payload|        kafka_timestamp|
+------------------------------------------------------------+-----------------------+
|{"HPAN": "999999XXXXXX0157", "ACCT1": 99000000157, "ACCT1...|2026-11-25 13:51:59.909|
|{"HPAN": "999999XXXXXX0027", "ACCT1": 99000000027, "ACCT1...|2026-11-25 13:52:00.011|
|{"HPAN": "999999XXXXXX0102", "ACCT1": 99000000102, "ACCT1...| 2026-11-25 13:52:00.12|
+------------------------------------------------------------+-----------------------+
only showing top 3 rows


🥈 SILVER LAYER SAMPLE (Cleaned & Deduplicated)
+-------+------------+--------+------+----------+
|ACCT_NO|ADDRESS_CITY|MERCHANT|ACTAMT|event_time|
+-------+------------+--------+------+----------+
+-------+------------+--------+------+----------+


🥇 GOLD LAYER (Business Aggregations)
+------------+----------+----------

In [13]:
print("=== 🥉 BRONZE LAYER SAMPLE ===")
spark.read.format("delta").load(f"{LAKEHOUSE_BASE}/bronze/transactions") \
    .select("raw_payload", "kafka_timestamp").show(3, truncate=50)

print("=== 🥈 SILVER LAYER SAMPLE ===")
spark.read.format("delta").load(f"{LAKEHOUSE_BASE}/silver/transactions") \
    .select("ACCT_NO", "ADDRESS_CITY", "MERCHANT", "ACTAMT", "event_time").show()

print("=== 🥇 GOLD LAYER (BUSINESS AGGREGATIONS) ===")
spark.read.format("delta").load(f"{LAKEHOUSE_BASE}/gold/merchant_summary") \
    .select("window_start", "ADDRESS_CITY", "MERCHANT", "RESP", "transaction_count", "total_amount").show()

=== 🥉 BRONZE LAYER SAMPLE ===
+--------------------------------------------------+-----------------------+
|                                       raw_payload|        kafka_timestamp|
+--------------------------------------------------+-----------------------+
|{"HPAN": "999999XXXXXX0157", "ACCT1": 990000001...|2026-11-25 13:51:59.909|
|{"HPAN": "999999XXXXXX0027", "ACCT1": 990000000...|2026-11-25 13:52:00.011|
|{"HPAN": "999999XXXXXX0102", "ACCT1": 990000001...| 2026-11-25 13:52:00.12|
+--------------------------------------------------+-----------------------+
only showing top 3 rows

=== 🥈 SILVER LAYER SAMPLE ===
+-------+------------+--------+------+----------+
|ACCT_NO|ADDRESS_CITY|MERCHANT|ACTAMT|event_time|
+-------+------------+--------+------+----------+
+-------+------------+--------+------+----------+

=== 🥇 GOLD LAYER (BUSINESS AGGREGATIONS) ===
+------------+------------+--------+----+-----------------+------------+
|window_start|ADDRESS_CITY|MERCHANT|RESP|transaction_coun